# LQR and linear MPC controller example for tracking ECI reference trajectory

In [ ]:
import numpy as np
from tqdm import tqdm
import time 


In [ ]:
from leo_gym.utils.utils import seed_all

seed_all(seed=1)

In [ ]:
from leo_gym.satellite.satellite_base import Satellite, SatelliteConfig, DynamicsConfig

days = 2
dt = 60

pv = [-5523165.96045516,190203.065417998,5184120.95174883,-4971.736483382,-193.42400685567,-5278.01680949887]

params_dyn_ideal=DynamicsConfig(
    flag_rtn_thrust=True,
    flag_mass_loss=False,
    flag_pert_moon=False,
    flag_pert_sun=False,
    flag_pert_srp=False,
    flag_pert_drag=False,
    flag_pert_irr_grav=False,
    eph_time_0=6.338304141847866e+08,
    m=150,
    f_max=18e-2,
    Isp=860,
    Ad=1.3,
    Cd=2.2,
    Cr=1.3,
    As=1.3,
    mf=136
)

sat_ideal_cfg = SatelliteConfig(
    delta_r_norm=0, 
    delta_v_norm=0, 
    rv0=np.array(pv),
    days=2,
    dt=dt,
    params_dyn=params_dyn_ideal
    
    )

sat_nom=Satellite(sat_ideal_cfg)

try:
    for loops in tqdm(range(int(24*days*60*60/dt)), desc="Processing"):
            
        sat_nom.sat_propagate(np.zeros(3))   
except KeyboardInterrupt:
    pass

In [ ]:
import numpy as np
import control
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt



def _DCM_eci2rtn(r:np.ndarray, v:np.ndarray)->np.ndarray:
    """
    :param r: ECI position vector
    :param B_d: ECI velocity vector
    
    :return: 3x3 matrix ECI -> RTN.

    """
    #Hill frame Transformation DCM from ECI to RTN
    r_norm = r / np.linalg.norm(r)
    v_norm = v / np.linalg.norm(v)
    # Orbit normal
    h_norm = (np.cross(r_norm, v_norm)
            / np.linalg.norm(np.cross(r_norm, v_norm)))
    # Orbit tangential
    t_norm = np.cross(h_norm, r_norm)
    return np.vstack((r_norm, t_norm, h_norm))


def cw_state_matrix(n:float):
    """
    Returns the state matrix A for the Clohessy-Wiltshire equations in the RTN frame.

    The state vector is defined as:
        x = [x, y, z, x_dot, y_dot, z_dot]^T,
    and the dynamics are given by:
        x_dot = A * x.

    The matrix A is:
        [  0      0      0      1      0      0  ]
        [  0      0      0      0      1      0  ]
        [  0      0      0      0      0      1  ]
        [ 3n^2    0      0      0     2n      0  ]
        [  0      0      0    -2n      0      0  ]
        [  0      0   -n^2      0      0      0  ]

    :param n: Mean motion of the reference orbit.

    :return: 6x6 state-space matrix for the CW equations.
    """
    A = np.array([
        [0,      0,     0,    1,    0,    0],
        [0,      0,     0,    0,    1,    0],
        [0,      0,     0,    0,    0,    1],
        [3*n**2, 0,     0,    0,   2*n,   0],
        [0,      0,     0,  -2*n,   0,     0],
        [0,      0,  -n**2,   0,    0,     0]
    ])
    return A

def cw_input_matrix(m):
    B = np.array([
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [1/m, 0, 0],
        [0, 1/m, 0],
        [0, 0, 1/m]
    ])
    return B

def finite_horizon_dlqr(A_d, B_d, Q, R, Qf, N):
    """
    Solves the finite-horizon discrete-time LQR problem.
    
    The dynamics are:
         x[k+1] = A_d x[k] + B_d u[k],
    and the cost function is
         J = x[N]^T Qf x[N] + sum_{k=0}^{N-1} (x[k]^T Q x[k] + u[k]^T R u[k]).
    
    Returns:
        K : list of feedback gain matrices [K[0], K[1], ..., K[N-1]].
        P : list of cost-to-go matrices [P[0], P[1], ..., P[N]].
    """
    n_states = A_d.shape[0]
    P = [None]*(N+1)
    K = [None]*N
    P[N] = Qf.copy()  # Terminal cost
    for k in range(N-1, -1, -1):
        S = R + B_d.T @ P[k+1] @ B_d
        K[k] = np.linalg.pinv(S) @ (B_d.T @ P[k+1] @ A_d)
        P[k] = Q + A_d.T @ P[k+1] @ A_d - A_d.T @ P[k+1] @ B_d @ K[k]
    return K, P



In [ ]:

params_dyn_ideal=DynamicsConfig(
    flag_rtn_thrust=True,
    flag_mass_loss=False,
    flag_pert_moon=False,
    flag_pert_sun=False,
    flag_pert_srp=False,
    flag_pert_drag=False,
    flag_pert_irr_grav=False,
    eph_time_0=6.338304141847866e+08,
    m=150,
    f_max=18e-2,
    Isp=860,
    Ad=1.3,
    Cd=2.2,
    Cr=1.3,
    As=1.3,
    mf=136
)

sat_ideal_cfg = SatelliteConfig(
    delta_r_norm=800, 
    delta_v_norm=0.1, 
    rv0=np.array(sat_nom.rvm_eci_states[0][:6]),
    days=2,
    dt=dt,
    params_dyn=params_dyn_ideal
    
    )

sat_real=Satellite(sat_ideal_cfg)



In [ ]:
import cvxpy as cp
import numpy as np

def finite_horizon_mpc(
    A_d:np.ndarray, 
    B_d:np.ndarray,
    Q:np.ndarray,
    R:np.ndarray,
    Qf:np.ndarray,
    N_c:int,
    x0:np.ndarray,
    u_max:float
    )->np.ndarray:
    
    """
    Solves the finite-horizon MPC problem as a QP.

    Dynamics:
        x[k+1] = A_d x[k] + B_d u[k],

    Cost function:
        J = x[N]^T Qf x[N] + sum_{k=0}^{N-1} (x[k]^T Q x[k] + u[k]^T R u[k])

    Subject to:
        -u_max <= u[k] <= u_max for all k.

    :param A_d: Discrete-time state transition matrix.
    :param B_d: Discrete-time input matrix.
    :param Q: State cost matrix.
    :param R: Control cost matrix.
    :param Qf: Terminal state cost matrix.
    :param N: Prediction horizon (number of discrete time steps).
    :param x0: Initial state vector.
    :param u_max: Maximum absolute value allowed for each control input element.

    :return: Tuple ``(u_opt, x_opt)`` where ``u_opt`` is the optimal control sequence over the horizon
            with shape ``(n_inputs, N)``, and ``x_opt`` is the optimal state trajectory with shape
            ``(n_states, N+1)``.
    """
    
    n_states = A_d.shape[0]
    n_inputs = B_d.shape[1]

    x = cp.Variable((n_states, N_c+1))
    u = cp.Variable((n_inputs, N_c))
    
    cost = 0
    constraints = []

    # Initial condition
    constraints += [x[:, 0] == x0]

    for k in range(N_c):
        cost += cp.quad_form(x[:, k], Q) + cp.quad_form(u[:, k], R)
        constraints += [x[:, k+1] == A_d @ x[:, k] + B_d @ u[:, k]]
        constraints += [cp.abs(u[:, k]) <= u_max]

    # Terminal cost
    cost += cp.quad_form(x[:, N_c], Qf)
    
    prob = cp.Problem(cp.Minimize(cost), constraints)
    prob.solve(solver=cp.CLARABEL)
    computation_time = prob.solver_stats.solve_time

    if prob.status in ["infeasible", "unbounded"]:
        raise ValueError("The MPC QP is infeasible or unbounded.")
    
    u_opt = u.value
    x_opt = x.value
    
    return u_opt, x_opt, computation_time


In [ ]:
class SatelliteControllerECI():
    """
    Satellite controller activation logic with minimum and lower position velocity bounds
    
    """
    def __init__(self, b_s, b_d):

        self.b_s = b_s
        self.b_d = b_d
        self.controller_flag = 0

    def controller(self, mpc_inputs, pos_errors, vel_errors):
        if np.linalg.norm(pos_errors)>=self.b_s[0] or np.linalg.norm(vel_errors)>=self.b_s[1] and self.controller_flag == 1:
            u ,_ , computation_time = finite_horizon_mpc(**mpc_inputs)
            u = u[:,0]
        elif np.linalg.norm(pos_errors)<=self.b_s[0] and np.linalg.norm(vel_errors)<=self.b_s[1]:
            u = np.zeros(3)
            self.controller_flag = 0
            
        elif np.linalg.norm(pos_errors)>=self.b_d[0] or np.linalg.norm(vel_errors)>=self.b_d[1] and self.controller_flag == 0:
            u ,_, computation_time = finite_horizon_mpc(**mpc_inputs)
            u = u[:,0]
            self.controller_flag = 1
            
        else:
            u = np.zeros(3)
            
        return u

In [ ]:
import os
from IPython.display import clear_output
import numpy as np
from scipy.integrate import solve_ivp
from scipy.signal import cont2discrete
from tqdm import tqdm
from typing import Tuple

def finite_horizon_dlqr(
    A_d:np.ndarray,
    B_d:np.ndarray,
    Q:np.ndarray, 
    R:np.ndarray,
    Qf:np.ndarray,
    N:int
    )-> Tuple[np.ndarray,np.ndarray]:
    """
    Solves the finite-horizon discrete-time LQR problem.
    
    The dynamics are:
         x[k+1] = A_d x[k] + B_d u[k],
    and the cost function is
         J = x[N]^T Qf x[N] + sum_{k=0}^{N-1} (x[k]^T Q x[k] + u[k]^T R u[k]).
    
    Returns:
        K : list of feedback gain matrices [K[0], K[1], ..., K[N-1]].
        P : list of cost-to-go matrices [P[0], P[1], ..., P[N]].
    """
    n_states = A_d.shape[0]
    P = [None]*(N+1)
    K = [None]*N
    P[N] = Qf.copy()  # Terminal cost
    for k in range(N-1, -1, -1):
        S = R + B_d.T @ P[k+1] @ B_d
        K[k] = np.linalg.inv(S) @ (B_d.T @ P[k+1] @ A_d)
        P[k] = Q + A_d.T @ P[k+1] @ A_d - A_d.T @ P[k+1] @ B_d @ K[k]
    return K, P

mu = 398600.4418  
r_ref = 7573454    
n = np.sqrt(mu / r_ref**3)

# Finite-horizon parameters
N_c = 35 # Number of discrete time steps in the horizon
Q = np.diag([0.1, 0.1, 0.1, 1e2, 1e2, 1e2])  # Stage cost
R = np.diag([1, 1, 1])*1e3 # Control cost
Qf = Q.copy() # Terminal cost


controller_logic = SatelliteControllerECI(b_s = [500,0.05], b_d = [1000, 0.5])

try:
    for loops in tqdm(range(int(24*days*60*60/dt)), desc="Processing",leave=True):
        
        clear_output(wait=True)

        current_state = sat_real.rvm_eci_states[-1]
        
        A = cw_state_matrix(n)
        B = cw_input_matrix(150)

        system = (A, B, np.eye(A.shape[0]), 0)
        A_d, B_d, _, _, _ = cont2discrete(system, dt)

        error_state_eci = sat_real.rvm_eci_states[-1] - sat_nom.rvm_eci_states[loops]

        pos_nom = sat_nom.rvm_eci_states[loops][:3]
        vel_nom = sat_nom.rvm_eci_states[loops][3:6]

        dcm = _DCM_eci2rtn(pos_nom, vel_nom)

        error_pos_rtn = dcm @ error_state_eci[:3]
        error_vel_rtn = dcm @ error_state_eci[3:6]
        error_state_rtn = np.concatenate((error_pos_rtn, error_vel_rtn))

        ### LQR Controller
        # K_seq, P_seq = finite_horizon_dlqr(A_d, B_d, Q, R, Qf, N_c)
        # u = -K_seq[0] @ error_state_rtn
        # u = np.clip(u, -18e-2, 18e-2)
        ###
        
        ### MPC Controller
        mpc_inputs = {
            "A_d" : A_d,
            "B_d" : B_d,
            "Q" : Q,
            "R" : R,
            "Qf" : Qf,
            "N_c" : N_c,
            "x0": error_state_rtn,
            "u_max" : 18e-2,
        }
        u = controller_logic.controller(mpc_inputs=mpc_inputs,pos_errors=error_pos_rtn,vel_errors=error_vel_rtn)
        ###
        
        x_next = A_d @ error_state_rtn + B_d @ u

        tqdm.write(f"""
        Station Keeping Step: {loops*dt/60:.2f} minutes
        ---------------------------------------
        ECI error L2 norm: {np.linalg.norm(sat_real.rvm_eci_states[-1][:3]-sat_nom.rvm_eci_states[loops][:3]):.4f} meters
        ECI vel error L2 norm: {np.linalg.norm(sat_real.rvm_eci_states[-1][3:6]-sat_nom.rvm_eci_states[loops][3:6]):.4f} meters/second

        Computation Time: seconds
        RTN thrust vector: {u} N
        Next error state (RTN): {np.linalg.norm(x_next[:3])}
        """)

        # time.sleep(0.1)

        sat_real.sat_propagate(u)

except KeyboardInterrupt:
    pass


In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

real_states = np.array(sat_real.rvm_eci_states)
nom_states  = np.array(sat_nom.rvm_eci_states)

N_c = min(len(real_states), len(nom_states))

rel_pos = real_states[:N_c, :3] - nom_states[:N_c, :3]
rel_vel = real_states[:N_c, 3:6] - nom_states[:N_c, 3:6]

pos_norm = np.linalg.norm(rel_pos, axis=1)
vel_norm = np.linalg.norm(rel_vel, axis=1)

time = np.linspace(0, N_c, N_c)

fig = make_subplots(
    rows=3, cols=3,
    specs=[
        [{}, {}, {}],          
        [{}, {}, {}],         
        [{"colspan": 2}, None, {}]  
    ],
    subplot_titles=[
        'Rel Pos Error X', 'Rel Pos Error Y', 'Rel Pos Error Z',
        'Rel Vel Error X', 'Rel Vel Error Y', 'Rel Vel Error Z',
        'Position Error Norm', 'Velocity Error Norm'
    ]
)

for i, axis in enumerate(['X', 'Y', 'Z']):
    fig.add_trace(
        go.Scatter(x=time, y=rel_pos[:, i], mode='lines', name=f'Rel Pos {axis}'),
        row=1, col=i+1
    )

for i, axis in enumerate(['X', 'Y', 'Z']):
    fig.add_trace(
        go.Scatter(x=time, y=rel_vel[:, i], mode='lines', name=f'Rel Vel {axis}'),
        row=2, col=i+1
    )

fig.add_trace(
    go.Scatter(x=time, y=pos_norm, mode='lines', name='Pos Error Norm'),
    row=3, col=1
)

fig.add_trace(
    go.Scatter(x=time, y=vel_norm, mode='lines', name='Vel Error Norm'),
    row=3, col=3
)

fig.update_layout(
    height=900,
    width=1200,
    title_text="Relative Position and Velocity Errors Over Time",
    showlegend=False
)

fig.show()
